In [ ]:
import os, glob, gc, re
import numpy as np
import xarray as xr
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

# ✅ 导入计算函数
try:
    from aostools_functions import ComputeEPfluxDiv
except ImportError:
    raise ImportError("请确保 'aostools_functions.py' 与此脚本在同一目录下！")

# ============================================================
# Data root: split variables into folders (from your SH output)
# ============================================================
ROOT_DIR = "/mnt/soclim0/public_data/weiji/B2000WCN001002"
U_DIR     = os.path.join(ROOT_DIR, "U")
V_DIR     = os.path.join(ROOT_DIR, "V")
T_DIR     = os.path.join(ROOT_DIR, "T")
OMEGA_DIR = os.path.join(ROOT_DIR, "OMEGA")

OUT_DIR     = os.path.join(ROOT_DIR, "EPflux_daily")
OUT_FILE_NC = os.path.join(OUT_DIR, "EPFLUX_206yr_Daily_Series_Full.nc")

os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- Settings (keep your old logic) ----------------
PLEV_STD_PA = np.array([
    10, 50, 100, 200, 300, 500, 1000, 2000, 3000, 5000, 7000, 10000, 15000,
    20000, 25000, 30000, 40000, 50000, 60000, 70000, 85000, 92500, 100000
], dtype=float)

LAT_RANGE   = (40.0, 80.0)
DO_UBAR     = False
WAVE        = -1
MAX_WORKERS = 32

# ---- Block-A style time shift ----
# physical file-year 105-210 (run002) will be shifted internally by +103 to become 104-209
SHIFT_RUN2_INTERNAL_YEAR_OFFSET = 103

# ---------------- Helpers ----------------
YEAR_RE = re.compile(r"\.cam\.h3\.(\d{4})\.")

def parse_year_from_filename(path: str) -> int:
    m = YEAR_RE.search(os.path.basename(path))
    if not m:
        raise ValueError(f"Cannot parse year from filename: {path}")
    return int(m.group(1))

def file_for(var_dir: str, year_int: int, var_name: str) -> str:
    y = f"{year_int:04d}"
    return os.path.join(var_dir, f"B2000WCN.sample.cam.h3.{y}.{var_name}.nc")

def compute_pressure_mid(ds: xr.Dataset) -> xr.DataArray:
    return ds["hyam"] * ds["P0"] + ds["hybm"] * ds["PS"]

def interp_profile_logp_4d(v_hyb, p_hyb, p_tgt_pa):
    p_tgt_pa = np.asarray(p_tgt_pa, float)

    def _interp_col(vcol, pcol):
        m = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        if m.sum() < 2:
            return np.full(p_tgt_pa.shape, np.nan, float)
        p_use, v_use = pcol[m], vcol[m]
        idx = np.argsort(p_use)
        return np.interp(
            np.log(p_tgt_pa),
            np.log(p_use[idx]),
            v_use[idx],
            left=np.nan,
            right=np.nan
        )

    dask_gufunc_kwargs = {"output_sizes": {"plev": len(p_tgt_pa)}}
    out = xr.apply_ufunc(
        _interp_col, v_hyb, p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True, dask="parallelized", output_dtypes=[float],
        dask_gufunc_kwargs=dask_gufunc_kwargs
    )
    return out.assign_coords(plev=("plev", p_tgt_pa))

def sel_latband(da, lat0=40., lat1=80., lat_name="lat"):
    lat = da[lat_name]
    dec = float(lat.values[0]) > float(lat.values[-1])
    slc = slice(lat1, lat0) if dec else slice(lat0, lat1)
    out = da.sel({lat_name: slc})
    if out.sizes.get(lat_name, 0) == 0:
        raise ValueError("Empty lat selection.")
    return out

def coslat_weighted_mean(da, lat_name="lat"):
    lat = da[lat_name]
    w = np.cos(np.deg2rad(lat))
    return da.weighted(w).mean(lat_name, skipna=True)

def shift_time_like_blockA(ds: xr.Dataset, offset_years: int) -> xr.Dataset:
    """
    完全照抄你 Block A 的做法：对 cftime 对象直接 year += offset
    """
    old_times = ds.time.values
    new_times = [t.replace(year=t.year + offset_years) for t in old_times]
    return ds.assign_coords(time=new_times)

# ---------------- Determine year list (Block-A physical selection) ----------------
def get_physical_years():
    u_files = sorted(glob.glob(os.path.join(U_DIR, "B2000WCN.sample.cam.h3.*.U.nc")))
    all_years = [parse_year_from_filename(f) for f in u_files]

    # Block A selection:
    years_001 = [y for y in all_years if 1 <= y <= 103]
    years_002 = [y for y in all_years if 105 <= y <= 210]  # skip 104 on purpose

    years = sorted(set(years_001 + years_002))
    return years

# ---------------- Single-year worker (parallel) ----------------
def process_one_year(year_int: int):
    """
    Read U/V/T/OMEGA of this physical file-year.
    If it's run002 (physical year>=105), shift internal cftime year by +103 (Block A).
    Then compute Fz(time, plev) area-mean 40–80N.
    """
    try:
        fU = file_for(U_DIR, year_int, "U")
        fV = file_for(V_DIR, year_int, "V")
        fT = file_for(T_DIR, year_int, "T")
        fW = file_for(OMEGA_DIR, year_int, "OMEGA")

        for fp in [fU, fV, fT, fW]:
            if not os.path.exists(fp):
                return f"⚠️ Missing file for year {year_int:04d}: {fp}"

        # use_cftime=True to match your Block A behavior
        time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)

        with xr.open_dataset(fU, decode_times=time_coder) as dsU, \
             xr.open_dataset(fV, decode_times=time_coder) as dsV, \
             xr.open_dataset(fT, decode_times=time_coder) as dsT, \
             xr.open_dataset(fW, decode_times=time_coder) as dsW:

            # --- Block A internal time shift for run002 physical years ---
            if year_int >= 105:
                dsU = shift_time_like_blockA(dsU, SHIFT_RUN2_INTERNAL_YEAR_OFFSET)
                dsV = shift_time_like_blockA(dsV, SHIFT_RUN2_INTERNAL_YEAR_OFFSET)
                dsT = shift_time_like_blockA(dsT, SHIFT_RUN2_INTERNAL_YEAR_OFFSET)
                dsW = shift_time_like_blockA(dsW, SHIFT_RUN2_INTERNAL_YEAR_OFFSET)

            # (keep your old logic) skip tiny years
            if dsU.sizes.get("time", 0) < 2:
                return None

            # 1) pressure mid
            p_mid = compute_pressure_mid(dsU)

            # 2) logp interp to std p
            u_std = interp_profile_logp_4d(dsU["U"], p_mid, PLEV_STD_PA)
            v_std = interp_profile_logp_4d(dsV["V"], p_mid, PLEV_STD_PA)
            t_std = interp_profile_logp_4d(dsT["T"], p_mid, PLEV_STD_PA)
            w_std = interp_profile_logp_4d(dsW["OMEGA"] / 100.0, p_mid, PLEV_STD_PA)

            # 3) to numpy
            u_np = u_std.transpose("time", "plev", "lat", "lon").values
            v_np = v_std.transpose("time", "plev", "lat", "lon").values
            t_np = t_std.transpose("time", "plev", "lat", "lon").values
            w_np = w_std.transpose("time", "plev", "lat", "lon").values
            lat_np = dsU["lat"].values

            # 4) EP flux (ep2 is Fz)
            _, ep2, _, _ = ComputeEPfluxDiv(
                lat=lat_np,
                pres=(PLEV_STD_PA / 100.0),  # hPa
                u=u_np, v=v_np, t=t_np, w=w_np,
                do_ubar=DO_UBAR, wave=WAVE
            )

            da_ep2 = xr.DataArray(
                ep2,
                coords={"time": dsU["time"], "plev": PLEV_STD_PA, "lat": lat_np},
                dims=("time", "plev", "lat"),
                name="Fz"
            )

            # 5) 40–80N coslat mean => (time, plev)
            da_sub = sel_latband(da_ep2, LAT_RANGE[0], LAT_RANGE[1], "lat")
            out = coslat_weighted_mean(da_sub).compute()
            out.name = "Fz"

        # cleanup
        del u_std, v_std, t_std, w_std, u_np, v_np, t_np, w_np, da_ep2, da_sub
        gc.collect()

        return out

    except Exception as e:
        return f"Error in year {year_int:04d}: {type(e).__name__}: {str(e)}"

# ---------------- Main ----------------
def main():
    years = get_physical_years()
    if not years:
        print(f"❌ No U files found in {U_DIR}")
        return

    print(f"🚀 EPflux daily: {len(years)} physical years (1–103 and 105–210), workers={MAX_WORKERS}")
    pool_results = []

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
        fut = {ex.submit(process_one_year, y): y for y in years}
        for f in tqdm(as_completed(fut), total=len(fut), desc="EPflux parallel"):
            res = f.result()
            if isinstance(res, xr.DataArray):
                pool_results.append(res)
            elif res is not None:
                print(f"⚠️ {res}")

    if not pool_results:
        print("❌ No data collected.")
        return

    print("📊 Finalizing: concat + sortby(time) + drop duplicate time ...")
    Fz_all = xr.concat(pool_results, dim="time").sortby("time")

    # drop duplicates (insurance)
    tvals = Fz_all["time"].values
    _, idx = np.unique(tvals, return_index=True)
    idx = np.sort(idx)
    Fz_all = Fz_all.isel(time=idx)

    ds_out = xr.Dataset({"Fz": Fz_all})
    ds_out.attrs = {
        "description": "EP Flux (Fz) FULL Daily Time Series (40–80N coslat mean)",
        "lat_range": f"{LAT_RANGE[0]}N-{LAT_RANGE[1]}N",
        "units": "m^2/s^2",
        "history": "Block-A style internal cftime year shift for run002 (+103 years); physical years 1-103 & 105-210; 104 skipped"
    }

    print(f"💾 Writing: {OUT_FILE_NC}")
    ds_out.to_netcdf(OUT_FILE_NC)
    print("✅ Done!")

if __name__ == "__main__":
    main()


In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# ============================================================
# Fig(f)-style plot: Fz standardized anomalies (blue-white-red, -4..4)
# Reference: 206-year Daily Full Series (NEW PATH, already 40–80N mean)
# Winter: 0007/11/01 - 0008/04/30 (model calendar)
# ============================================================

# ---------------- paths ----------------
# 你的 hindcast / 23-year 信号文件仍然在旧目录（保持不变）
RESULT_DIR = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result"

EP_NC  = f"{RESULT_DIR}/EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc"
EHF_NC = f"{RESULT_DIR}/EHF_BWCN002_vTprime_ALL_LAT_stdplev_time_lat.nc"

# ✅ 新的 206-yr EPflux daily 输出目录（来自你新 pipeline）
REF_ROOT   = "/mnt/soclim0/public_data/weiji/B2000WCN001002"
REF_DIR    = os.path.join(REF_ROOT, "EPflux_daily")
CLIM_NC    = os.path.join(REF_DIR, "EPFLUX_206yr_Daily_Series_Full.nc")

# ✅ 输出图片路径（也放到新 EPflux_daily 目录里）
OUT_FIG = os.path.join(RESULT_DIR, "Fig_f_Fz_std_anom_206yr_FullRef.png")

# ---------------- settings ----------------
t0, t1 = "0007-11-01", "0008-04-30"  # 目标窗口（包含4月）
LAT0, LAT1 = 40.0, 80.0
ZLIM = 4.0

# ---------------- helpers ----------------
def sel_latband(da, lat0=40., lat1=80., lat_name="lat"):
    lat = da[lat_name]
    dec = float(lat.values[0]) > float(lat.values[-1])
    slc = slice(lat1, lat0) if dec else slice(lat0, lat1)
    out = da.sel({lat_name: slc})
    if out.sizes.get(lat_name, 0) == 0:
        raise ValueError("Empty lat selection.")
    return out

def coslat_weighted_mean(da, lat_name="lat"):
    lat = da[lat_name]
    w = np.cos(np.deg2rad(lat))
    return da.weighted(w).mean(lat_name, skipna=True)

# ---------------- load & pre-process ----------------
print("📖 Loading datasets...")
ds_ep   = xr.open_dataset(EP_NC)
ds_ehf  = xr.open_dataset(EHF_NC)
ds_full = xr.open_dataset(CLIM_NC)

# 信号场（向上为正）
Fz_sig_up = -ds_ep["EP2_cart"]
vT_sig    = ds_ehf["EHF_vTprime_stdplev"]

# ✅ 新输出：ds_full["Fz"] 已经是 40–80N 区域平均后的 (time, plev)
# 仍然按你原逻辑翻号，使“向上为正”
Fz_ref_up = -ds_full["Fz"]

# ---------------- signal selection ----------------
Fz_win = Fz_sig_up.sel(time=slice(t0, t1))
vT_win = vT_sig.sel(time=slice(t0, t1))

# 信号场区域平均：40–80N
Fz_40_80 = coslat_weighted_mean(sel_latband(Fz_win, LAT0, LAT1))
vT_40_80 = coslat_weighted_mean(sel_latband(vT_win, LAT0, LAT1))

nt = Fz_40_80.sizes["time"]
x = np.arange(nt)
time_str = np.array([str(tt)[:10] for tt in Fz_40_80["time"].values])

# ---------------- standardize using 206-yr Climatology ----------------
print("📊 Calculating standardized anomalies using 206-yr reference...")

# 1) 从 206 年全序列计算每日气候态 (dayofyear)
# Fz_ref_up: (time, plev) => clim_mu/clim_sd: (dayofyear, plev)
clim_mu = Fz_ref_up.groupby("time.dayofyear").mean("time")
clim_sd = Fz_ref_up.groupby("time.dayofyear").std("time")

# 2) 匹配信号场每一天对应的气候态
sig_doy = Fz_40_80.time.dt.dayofyear
mu_matched = clim_mu.sel(dayofyear=sig_doy)
sd_matched = clim_sd.sel(dayofyear=sig_doy).where(lambda x: x > 1e-12)

# 3) 标准化距平：结果保持 (time, plev)
Fz_stdzd_vals = (Fz_40_80.values - mu_matched.values) / sd_matched.values
Fz_stdzd = xr.DataArray(Fz_stdzd_vals, coords=Fz_40_80.coords, dims=Fz_40_80.dims)

# ---------------- plotting arrays ----------------
plev = Fz_stdzd["plev"].values
Z_Fz = Fz_stdzd.transpose("plev", "time").values
Z_vT = vT_40_80.transpose("plev", "time").values

# ---------------- plot ----------------
print("🎨 Plotting...")
fig, ax = plt.subplots(figsize=(14, 4.8))

levels = np.linspace(-ZLIM, ZLIM, 17)
cmap = plt.get_cmap("RdBu_r")
norm = TwoSlopeNorm(vmin=-ZLIM, vcenter=0.0, vmax=ZLIM)

cf = ax.contourf(x, plev, Z_Fz, levels=levels, cmap=cmap, norm=norm, extend="both")

# 叠加 v'T' = 0 等值线 (虚线)
ax.contour(x, plev, Z_vT, levels=[0.0], colors="k", linestyles="--", linewidths=1.3)

# 坐标轴设置
ax.set_yscale("log")
ax.invert_yaxis()

# Y 轴刻度 (Pa 转 hPa)
yticks_pa = np.array([100000, 70000, 50000, 30000, 20000, 10000, 7000, 5000, 3000, 2000, 1000, 500, 300, 200, 100, 50, 10])
yticks_pa = yticks_pa[(yticks_pa >= float(plev.min())) & (yticks_pa <= float(plev.max()))]
ax.set_yticks(yticks_pa)
ax.set_yticklabels([f"{int(p/100)}" for p in yticks_pa])
ax.set_ylabel("Pressure (hPa)")

# X 轴时间刻度
tick_idx = np.linspace(0, nt-1, 8, dtype=int)
ax.set_xticks(tick_idx)
ax.set_xticklabels(time_str[tick_idx])
ax.set_xlabel("Model Date")

title_str = "40–80°N $F_z$ Std Anomaly (Ref: 206-yr Full Daily Climatology)\n"
title_str += r"Winter 0007/0008 | Dashed: $\overline{v'T'}=0$"
ax.set_title(title_str)

cbar = fig.colorbar(cf, ax=ax, pad=0.02)
cbar.set_label("Standardized Anomaly ($\sigma$)")
cbar.set_ticks([-4, -2, 0, 2, 4])

plt.tight_layout()

# 保存
os.makedirs(os.path.dirname(OUT_FIG), exist_ok=True)
plt.savefig(OUT_FIG, dpi=200, bbox_inches="tight")
print(f"✅ Success! Figure saved to: {OUT_FIG}")
plt.show()

# 释放内存
ds_ep.close(); ds_ehf.close(); ds_full.close()
